# Aula 15 - Notebook: Simulação de Falhas e Desvio Automático na Linha de Paçoca

Neste notebook simulamos uma contingência em tempo real na planta de processamento de amendoim.
O cenário considera uma falha/vazamento em um trecho de transferência após a torra e o recálculo
dinâmico de uma rota alternativa pelo sistema supervisório.

A aplicação representa a lógica de um SCADA integrado aos CLPs: ao detectar a indisponibilidade
de um trecho, o sistema isola a conexão e procura uma rota alternativa para manter o fluxo do
produto sem interromper toda a linha.

In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(
        f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols)
    )
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join(
        "-" * larguras[j] for j in range(len(rotulos_cols))
    )
    linhas = [header, divisor]

    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))

    return "\n".join(linhas)


class GrafoProcesso:
    """Representa os equipamentos e trechos de transferência da planta."""

    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)

        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]

        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0

        self.arestas_detalhes = []

    def adicionar_trecho(self, origem, destino, comprimento_m, tag_atuador,
                         tipo="Esteira/Transferência"):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]

        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m

        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Tag": tag_atuador,
            "Tipo": tipo
        })


def criar_linha_pacoca():
    # Equipamentos principais e pontos de transferência da planta.
    nos = [
        "RECEPCAO",
        "LIMPEZA",
        "SECAGEM",
        "SILO",
        "SELECAO_OPTICA",
        "TORRA",
        "DESPEL.",
        "MOAGEM",
        "DOSAGEM",
        "PRENSA",
        "EMBALAGEM",
        "PULMAO_TORRA"
    ]

    g = GrafoProcesso(nos)

    # Fluxo principal
    g.adicionar_trecho("RECEPCAO", "LIMPEZA", 12.0, "CV-101")
    g.adicionar_trecho("LIMPEZA", "SECAGEM", 18.0, "CV-102")
    g.adicionar_trecho("SECAGEM", "SILO", 15.0, "CV-103")
    g.adicionar_trecho("SILO", "SELECAO_OPTICA", 20.0, "CV-104")
    g.adicionar_trecho("SELECAO_OPTICA", "TORRA", 16.0, "CV-105")
    g.adicionar_trecho("TORRA", "DESPEL.", 14.0, "CV-106")
    g.adicionar_trecho("DESPEL.", "MOAGEM", 12.0, "CV-107")
    g.adicionar_trecho("MOAGEM", "DOSAGEM", 10.0, "CV-108")
    g.adicionar_trecho("DOSAGEM", "PRENSA", 8.0, "CV-109")
    g.adicionar_trecho("PRENSA", "EMBALAGEM", 11.0, "CV-110")

    # Rota alternativa hipotética por pulmão intermediário.
    # Ela não elimina nenhuma etapa de processo; apenas contorna um trecho de transferência.
    g.adicionar_trecho("TORRA", "PULMAO_TORRA", 9.0, "XV-201",
                        "Desvio pneumático")
    g.adicionar_trecho("PULMAO_TORRA", "DESPEL.", 10.0, "CV-201",
                        "Esteira alternativa")

    return g


import time
import heapq
from typing import Dict, Any, Tuple, Optional, Set, List


class RoteadorDijkstra:
    def __init__(self, grafo: GrafoProcesso):
        self.g = grafo

    def calcular_menor_caminho(
        self,
        origem: str,
        destino: str,
        bloqueios: Optional[Set[str]] = None
    ) -> Tuple[float, List[str]]:

        if bloqueios is None:
            bloqueios = set()

        if origem in bloqueios or destino in bloqueios:
            return float('inf'), []

        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}

        dist[origem] = 0.0
        heap = [(0.0, origem)]

        while heap:
            d_u, u = heapq.heappop(heap)

            if d_u > dist[u]:
                continue

            if u == destino:
                break

            u_idx = self.g.v_to_idx[u]

            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]

                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso

                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))

        caminho = []
        atual = destino

        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]

        caminho.reverse()

        if caminho and caminho[0] == origem:
            return dist[destino], caminho

        return float('inf'), []


class SistemaDesvioAutomatico:
    def __init__(self, grafo: GrafoProcesso):
        self.grafo = grafo
        self.roteador = RoteadorDijkstra(grafo)

    def tratar_falha(
        self,
        origem_falha: str,
        destino_falha: str,
        origem_fluxo: str,
        destino_fluxo: str
    ) -> Dict[str, Any]:

        t0 = time.perf_counter()

        u = self.grafo.v_to_idx[origem_falha]
        v = self.grafo.v_to_idx[destino_falha]

        # Isolamento lógico do trecho com falha.
        self.grafo.adj_pesos[u][v] = float('inf')
        self.grafo.adj_binaria[u][v] = 0

        novo_custo, nova_rota = self.roteador.calcular_menor_caminho(
            origem_fluxo,
            destino_fluxo
        )

        t_ms = (time.perf_counter() - t0) * 1000.0

        return {
            "Trecho_Isolado": f"{origem_falha} -> {destino_falha}",
            "Nova_Rota_Ativa": " -> ".join(nova_rota),
            "Comprimento_Total_m": f"{novo_custo:.1f}",
            "Tempo_Decisão_ms": f"{t_ms:.4f}"
        }


# Instanciação da planta
rede = criar_linha_pacoca()

# Cenário: falha no trecho principal TORRA -> DESPEL.
desviador = SistemaDesvioAutomatico(rede)

resultado = desviador.tratar_falha(
    "TORRA",
    "DESPEL.",
    "RECEPCAO",
    "EMBALAGEM"
)

print("=== RELATÓRIO DE DESVIO AUTOMÁTICO — LINHA DE PAÇOCA ===")
print(formatar_tabela([resultado]))

# A rota deve utilizar o pulmão intermediário como desvio.
assert "PULMAO_TORRA" in resultado["Nova_Rota_Ativa"]
assert "EMBALAGEM" in resultado["Nova_Rota_Ativa"]
